[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/94-zep-memory-server/zep_memory_workbook.ipynb)

# 94 · Zep Memory Server — Persistent Memory with Auto-Summarization
## Batteries-Included Memory: Auto-Summarize, Entity Extraction, Temporal Decay
⏱ ~90 min

Zep is a managed memory service for AI agents. Unlike Redis (which stores raw key-value pairs) or in-memory lists, Zep automatically summarizes conversation history when it grows long, extracts named entities, and applies temporal decay so that old facts matter less over time. This workbook builds the **load_memory → respond → save_memory** LangGraph pipeline using the Zep Cloud API.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **What Zep does** — auto-summarization, entity extraction, temporal decay |
| 2 | **Redis vs. Zep** — tradeoff table and motivation |
| 3 | **Zep API basics** — add_session, add, get, memory.context |
| 4 | **Three-node graph** — load_memory → respond → save_memory |
| 5 | **Full session** — seed conversation, query, observe auto-compressed context |
| ★ | **Exercises + Answer Key** |

> **Note:** requires both `OPENAI_API_KEY` and `ZEP_API_KEY`.
> Get a free Zep Cloud key at https://app.getzep.com


In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv", "zep-cloud"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["ZEP_API_KEY"] = userdata.get("ZEP_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

openai_ok = bool(os.environ.get("OPENAI_API_KEY", "")) and os.environ.get("OPENAI_API_KEY","").startswith("sk-")
zep_ok = bool(os.environ.get("ZEP_API_KEY", ""))
print(f"OPENAI_API_KEY: {openai_ok}")
print(f"ZEP_API_KEY:    {zep_ok}")
if not openai_ok:
    print("  ACTION: set OPENAI_API_KEY (sk-...)")
if not zep_ok:
    print("  ACTION: get a free Zep Cloud key at https://app.getzep.com")


---
## Part 1 — What Zep Does: Auto-Summarization, Entity Extraction, Temporal Decay

Zep is a **managed memory service** designed for long-running AI conversations. It solves three problems that naive memory approaches cannot:

### 1. Auto-Summarization
When a session accumulates more than a threshold of messages (typically ~10), Zep automatically summarizes the older portion into a compressed summary. The most recent turns are kept verbatim. The result is a `memory.context` string that is always bounded — regardless of how long the conversation runs.

### 2. Entity Extraction
As messages flow in, Zep extracts named entities (people, organizations, places, products) and tracks them across the conversation. This gives agents a structured view of *who* and *what* the conversation is about, without any extra prompting.

### 3. Temporal Decay
Zep applies time-weighted relevance to facts. A fact from last week matters less than a fact from this minute. This is configurable and mirrors how human memory works — recent context is more salient than old context.

### Data Structures
The cell below illustrates what these structures look like, without making any API calls:


In [ ]:
# Illustrate Zep's memory structure without API calls
# Zep returns a "memory context" — compressed history + recent raw messages

# What raw history looks like before Zep processes it (10 turns):
raw_history = [
    {"role": "user", "content": "Hi, I'm Sarah. I'm a software engineer at Acme Corp."},
    {"role": "assistant", "content": "Nice to meet you, Sarah!"},
    {"role": "user", "content": "I'm working on a Python project."},
    {"role": "assistant", "content": "What kind of project?"},
    {"role": "user", "content": "A data pipeline for processing CSV files."},
    {"role": "assistant", "content": "Got it. Are you using pandas?"},
    {"role": "user", "content": "Yes, and also dask for larger files."},
    {"role": "assistant", "content": "Good choice for scaling."},
    {"role": "user", "content": "Now I need help with error handling."},
    {"role": "assistant", "content": "Sure, what specific errors are you seeing?"}
]

# What Zep returns after auto-summarization (old turns -> summary, recent turns kept raw):
zep_context_mock = {
    "summary": "Sarah is a software engineer at Acme Corp building a Python data pipeline for CSV processing using pandas and dask.",
    "recent_messages": [
        {"role": "user", "content": "Yes, and also dask for larger files."},
        {"role": "assistant", "content": "Good choice for scaling."},
        {"role": "user", "content": "Now I need help with error handling."},
        {"role": "assistant", "content": "Sure, what specific errors are you seeing?"}
    ],
    "entities": ["Sarah", "Acme Corp", "Python", "pandas", "dask"]
}

print("Raw history: 10 turns =", sum(len(m["content"]) for m in raw_history), "chars")
print()
print("After Zep processing:")
print("  Summary:", zep_context_mock["summary"])
print("  Recent messages:", len(zep_context_mock["recent_messages"]), "turns kept raw")
print("  Entities extracted:", zep_context_mock["entities"])
print()
summary_chars = len(zep_context_mock["summary"])
recent_chars = sum(len(m["content"]) for m in zep_context_mock["recent_messages"])
print(f"Context size: {summary_chars + recent_chars} chars (vs {sum(len(m['content']) for m in raw_history)} raw)")
print(f"Compression: {(1 - (summary_chars + recent_chars) / sum(len(m['content']) for m in raw_history)):.0%} reduction")


---
## Part 2 — Redis vs. Zep Tradeoff Table

| Feature | Redis (raw list) | Zep Cloud |
|---------|-----------------|----------|
| **Auto-summarization** | No — you manage it | Yes — automatic when threshold hit |
| **Entity extraction** | No | Yes — names, orgs, places tracked |
| **Temporal decay** | No | Yes — configurable time-weighted relevance |
| **Storage** | In-memory / persistent (configurable) | Managed cloud (Zep servers) |
| **Query capability** | Key lookup, basic search | Semantic search over memory |
| **Setup complexity** | Medium (self-hosted) or Low (Redis Cloud) | Low (API key only) |
| **Cost** | Infra cost or Redis Cloud pricing | Free tier + usage-based |
| **Best for** | Low-latency caching, custom memory logic | Batteries-included long-horizon conversations |

### When to choose which

- **Choose Redis** when you need fine-grained control over what to store, sub-millisecond retrieval, or you are building custom memory logic that does not fit Zep's model.
- **Choose Zep** when you want a complete conversation memory layer that just works — with summarization, entity tracking, and temporal decay handled for you.

The cell below demonstrates the problem that motivates Zep: naive list-based memory grows without bound.


In [ ]:
# The naive Redis/list memory approach: just append everything
# Problem: context grows unbounded; token limit exceeded

naive_memory = []

def naive_add(role: str, content: str):
    naive_memory.append({"role": role, "content": content})

def naive_get_context(max_turns: int = 10) -> str:
    recent = naive_memory[-max_turns:]
    return "\n".join(f"{m['role']}: {m['content']}" for m in recent)

# Simulate 20 turns
for i in range(10):
    naive_add("user", f"Turn {i+1}: question about topic {i+1} with some details about the project")
    naive_add("assistant", f"Answer {i+1}: here is my response to the question about topic {i+1}")

print(f"Naive memory after 20 turns: {len(naive_memory)} messages")
ctx = naive_get_context(20)
print(f"Context tokens (approx): {len(ctx.split())}")
print()
print("Problem: naive memory grows linearly with conversation length.")
print("At 100 turns: ~5000 tokens just for history context.")
print("At 1000 turns: the context window is exceeded and old turns are lost anyway.")
print()
print("Zep's approach: auto-summarize when threshold is reached.")
print("Result: context stays bounded at ~500-1000 tokens regardless of conversation length.")


---
## Part 3 — Zep API Basics: add_session, add, get, memory.context

The Zep Cloud Python SDK exposes a clean interface:

| Method | Purpose |
|--------|---------|
| `Zep(api_key=...)` | Create a client connected to Zep Cloud |
| `client.memory.add_session(session_id, user_id)` | Create a new conversation session |
| `client.memory.add(session_id, messages=[...])` | Add messages (list of `Message` objects) to a session |
| `client.memory.get(session_id)` | Retrieve the session's memory object |
| `memory.context` | The compressed context string — ready to inject into a system prompt |
| `memory.messages` | The N most recent raw messages kept verbatim |

### Message format
```python
from zep_cloud.types import Message
Message(role="user", role_type="user", content="Hello!")
Message(role="assistant", role_type="assistant", content="Hi there!")
```

The `role` field is the display name; `role_type` must be `"user"` or `"assistant"` for Zep to understand the turn structure.


In [ ]:
import os
import uuid

ZEP_API_KEY = os.environ.get("ZEP_API_KEY", "")

try:
    from zep_cloud.client import Zep
    from zep_cloud.types import Message
    ZEP_IMPORT_OK = True
except ImportError:
    ZEP_IMPORT_OK = False

if not ZEP_API_KEY or not ZEP_IMPORT_OK:
    print("ZEP_API_KEY not set or zep-cloud not installed — skipping live API demo.")
    print("Set ZEP_API_KEY and install zep-cloud to run this cell.")
else:
    # Create client — connects to Zep Cloud
    client = Zep(api_key=ZEP_API_KEY)

    # Create a unique session for this demo
    SESSION_ID = f"demo-{uuid.uuid4().hex[:8]}"
    USER_ID = "demo-user"

    print(f"Creating session: {SESSION_ID}")
    client.memory.add_session(session_id=SESSION_ID, user_id=USER_ID)

    # Add some seed messages
    seed_messages = [
        Message(role="user", role_type="user", content="Hi, I'm Alex. I work in machine learning."),
        Message(role="assistant", role_type="assistant", content="Nice to meet you, Alex!"),
        Message(role="user", role_type="user", content="I'm building a recommendation system."),
        Message(role="assistant", role_type="assistant", content="What kind of recommendations?"),
    ]

    client.memory.add(session_id=SESSION_ID, messages=seed_messages)
    print(f"Added {len(seed_messages)} messages to session.")

    # Retrieve memory
    import time; time.sleep(1)  # brief wait for Zep to process
    memory = client.memory.get(session_id=SESSION_ID)
    print(f"\nmemory.context: {memory.context or '(not yet summarized — need more turns)'}")
    print(f"memory.messages count: {len(memory.messages) if memory.messages else 0}")

    print(f"\nSession ID saved for next cells: {SESSION_ID}")


---
## Part 4 — The Three-Node Graph: load_memory → respond → save_memory

The core LangGraph pattern for Zep memory is a three-node pipeline:

```
load_memory ──► respond ──► save_memory
```

### Node responsibilities

| Node | What it does |
|------|--------------|
| `load_memory` | Calls `client.memory.get(session_id)` and stores `memory.context` in state |
| `respond` | Runs the LLM with the memory context injected into the system prompt |
| `save_memory` | Calls `client.memory.add(session_id, messages=[user_msg, assistant_msg])` to persist the turn |

### Why this order?
- **Load first**: the LLM needs the compressed history before generating a response
- **Save last**: we only store confirmed turns — not drafts or errors
- **Stateless nodes**: each node reads from and writes to the LangGraph state dict, keeping logic clean

### State schema
```python
class ZepMemoryState(TypedDict):
    session_id: str      # Zep session identifier
    user_message: str    # current user input
    memory_context: str  # loaded from Zep (injected into system prompt)
    response: str        # generated by the LLM
```


In [ ]:
import os
import uuid
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

try:
    from zep_cloud.client import Zep
    from zep_cloud.types import Message as ZepMessage
    ZEP_AVAILABLE = bool(os.environ.get("ZEP_API_KEY"))
except ImportError:
    ZEP_AVAILABLE = False

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

class ZepMemoryState(TypedDict):
    session_id: str
    user_message: str
    memory_context: str    # loaded from Zep
    response: str          # generated by LLM

BASE_SYSTEM = "You are a helpful assistant. You have access to conversation history below."

def load_memory_node(state: ZepMemoryState) -> dict:
    """Load memory context from Zep for this session."""
    if not ZEP_AVAILABLE:
        print("  [Mock] load_memory: ZEP_API_KEY not set — returning empty context")
        return {"memory_context": ""}
    client = Zep(api_key=os.environ["ZEP_API_KEY"])
    try:
        memory = client.memory.get(session_id=state["session_id"])
        ctx = memory.context or ""
        print(f"  Loaded {len(ctx)} chars of memory context")
        return {"memory_context": ctx}
    except Exception as e:
        print(f"  load_memory error: {e}")
        return {"memory_context": ""}

def respond_node(state: ZepMemoryState) -> dict:
    """Generate a response using memory context as part of the system prompt."""
    system_content = BASE_SYSTEM
    if state["memory_context"]:
        # Inject compressed memory context — Zep handles summarization
        system_content += f"\n\nConversation history:\n{state['memory_context']}"
    resp = llm.invoke([
        SystemMessage(content=system_content),
        HumanMessage(content=state["user_message"])
    ])
    print(f"  Response: {resp.content[:80]}...")
    return {"response": resp.content.strip()}

def save_memory_node(state: ZepMemoryState) -> dict:
    """Save this turn to Zep memory for the next invocation."""
    if not ZEP_AVAILABLE:
        print("  [Mock] save_memory: ZEP_API_KEY not set — skipping")
        return {}
    client = Zep(api_key=os.environ["ZEP_API_KEY"])
    messages = [
        ZepMessage(role="user", role_type="user", content=state["user_message"]),
        ZepMessage(role="assistant", role_type="assistant", content=state["response"])
    ]
    try:
        client.memory.add(session_id=state["session_id"], messages=messages)
        print(f"  Saved 2 messages to session {state['session_id']}")
    except Exception as e:
        print(f"  save_memory error: {e}")
    return {}

print("Three nodes defined: load_memory_node, respond_node, save_memory_node")


---
## Part 5 — Full Session: Seed Conversation, Query, Observe Auto-Compressed Context

Now we compile the full pipeline and run a multi-turn session. The key observation:

- **Early turns**: `memory.context` may be empty or minimal — not enough history to summarize
- **After ~10 turns**: Zep automatically summarizes older turns; `memory.context` grows but stays bounded
- **The last cell in each turn**: the `load_memory_node` picks up whatever Zep has stored so far

Run all 4 turns below, then inspect `final_memory.context` to see the compressed representation.


In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(ZepMemoryState)
g.add_node("load_memory", load_memory_node)
g.add_node("respond", respond_node)
g.add_node("save_memory", save_memory_node)
g.set_entry_point("load_memory")
g.add_edge("load_memory", "respond")
g.add_edge("respond", "save_memory")
g.add_edge("save_memory", END)
app = g.compile()

# Use existing session if available, or create new one
if ZEP_AVAILABLE:
    try:
        SESSION_ID  # use session from Part 3 if it exists
    except NameError:
        SESSION_ID = f"demo-{uuid.uuid4().hex[:8]}"
        client = Zep(api_key=os.environ["ZEP_API_KEY"])
        client.memory.add_session(session_id=SESSION_ID, user_id="demo-user")
        print(f"Created new session: {SESSION_ID}")
else:
    SESSION_ID = "mock-session"

conversation = [
    "Hi, I'm Jordan. I work as a data scientist at a healthcare startup.",
    "I'm building a model to predict patient readmission rates.",
    "What features are typically most predictive for this?",
    "Can you summarize what we've discussed so far?"
]

print(f"Running {len(conversation)}-turn conversation (session: {SESSION_ID})\n")

for turn_num, message in enumerate(conversation, 1):
    print(f"Turn {turn_num}: {message}")
    result = app.invoke({
        "session_id": SESSION_ID,
        "user_message": message,
        "memory_context": "",
        "response": ""
    })
    print(f"  Assistant: {result['response'][:150]}")
    print()

# Show final memory state
if ZEP_AVAILABLE:
    import time; time.sleep(2)
    client = Zep(api_key=os.environ["ZEP_API_KEY"])
    final_memory = client.memory.get(session_id=SESSION_ID)
    print("Final memory.context:")
    print(final_memory.context or "(context builds after ~10 messages)")


---
## Exercises

### Exercise (a) — Observe Context Compression
Add 10 more seed turns using `app.invoke()` (the `extra_turns` list below is ready to use). After each batch, call `client.memory.get(session_id=SESSION_ID)` and inspect `memory.context`. At what turn count does Zep start summarizing?

### Exercise (b) — Measure the Compression Ratio
After adding all turns, compare:
- Total characters in all messages you have sent (`sum of len(msg) for all turns`)
- `len(memory.context)` from Zep

Compute the compression ratio: `context_size / total_raw_size`. What percentage of the raw text does Zep's context represent?


In [ ]:
# Exercise (a): Add 10 more seed turns and observe compression
# Run these turns through app.invoke() and then check memory.context

extra_turns = [
    "What libraries should I use for this ML project?",
    "Should I use XGBoost or a neural network for tabular data?",
    "How should I handle class imbalance in the training data?",
    "What's a good train/validation/test split ratio?",
    "How do I prevent data leakage when using patient data?",
    "Should I use SHAP or LIME for feature importance?",
    "How do I validate that my model generalizes to new hospitals?",
    "What metrics should I report beyond AUC?",
    "How do I deploy this model to production safely?",
    "What monitoring should I set up post-deployment?"
]

print("Exercise (a): run these 10 extra turns through app.invoke()")
print("Then check memory.context — should see auto-summarization kick in")
print()
for i, turn in enumerate(extra_turns[:3]):  # show first 3
    print(f"  Turn: {turn}")
print(f"  ... and {len(extra_turns)-3} more")

print()
# Exercise (b): compare raw history size to compressed context size
print("Exercise (b): Compression comparison")
print("After adding 14+ turns total:")
print("  len(memory.messages) = recent raw messages kept by Zep")
print("  len(memory.context) = compressed context (summary + recent raw)")
print("  Compression ratio = context_size / total_raw_size")
print()
print("TODO: measure before/after sizes and compute the ratio")


---
## Answer Key

### Exercise (a)
Zep's auto-summarization threshold is typically ~10 messages. After adding 10+ turns, `memory.context` will contain a generated summary of the older portion. The `memory.messages` array will only contain the N most recent turns, keeping the raw context bounded regardless of conversation length.

### Exercise (b)
A conversation with 20 turns (avg 50 tokens/turn = ~1000 tokens total raw) typically compresses to ~200-300 tokens in `memory.context` — a **70-80% reduction**. The summary preserves semantic meaning (who the user is, what they're building) while discarding verbatim exchanges. The compression ratio will vary based on message verbosity and Zep's summarization model.

---

## Zep vs. Mem0 (example 67)

| | Zep | Mem0 |
|-|-----|------|
| **Memory management** | Automatic — Zep decides what to summarize | Manual — you call `mem.add()` to store specific facts |
| **Retrieval** | `memory.context` — one compressed string | `mem.search()` — semantic search returns relevant facts |
| **Control** | Low (Zep manages the policy) | High (you choose what to remember) |
| **Best for** | Full conversation summaries that just work | Selective fact storage with semantic recall |

**Choose Mem0** when you want to hand-pick what to remember — e.g., "remember that the user prefers dark mode".

**Choose Zep** when you want a complete conversation summary that handles compression automatically — ideal for long support or tutoring conversations.
